In [4]:
"""
HalDetect Pipeline — Extraction, Deduplication, Classification (v2)
CSE 700 · University of Chittagong · 2026

Pipeline stages implemented here:
    1. Paragraph generation (LLM)
    2. Atomization (LLM) — with subjectivity filter built into prompt
    3. Deduplication (Sentence-BERT, τ = 0.90)
    4. Drift check (Sentence-BERT cosine sim vs source paragraph)
    5. Classification (LLM, few-shot, temperature=0, seed=42)
    6. Structured JSON extraction per claim (LLM)
    7. CSV output for manual gold labeling

Verifiers (TEMPORAL, ENTITY_ATTRIBUTE, CITATION, QUANTITATIVE, RELATIONAL)
are separate modules — not implemented here.

CHANGES FROM v1 (five fixes, all instrumentation/prompt changes — no behavior
change to what a claim ultimately gets classified/extracted as):

1. CLASSIFICATION FAILURE VISIBILITY. v1's classify_claim() silently returned
   "RELATIONAL" whenever the LLM's raw output didn't contain one of the five
   category words verbatim — indistinguishable from a genuine RELATIONAL
   classification. Fix: on failure, re-prompt once; if still unmatched, tag the
   row with classification_fallback_used=True and evidence in
   classification_raw_output, rather than silently defaulting.

2. EXTRACTION FAILURE VISIBILITY. v1's extract_structure() returned
   {subject: None, property: None, value: None} on a JSONDecodeError —
   indistinguishable from a genuinely empty extraction. Fix: tag the row with
   extraction_failed=True instead of silently returning nulls.

3. CITATION DUAL-EMIT PAIRING. v1's atomize() had no mechanism to link a
   source-attribution claim back to its embedded-fact sibling when the
   atomization prompt dual-emits both (per the prompt's own rule). Fix: the
   atomization prompt now asks the LLM to tag both halves of a dual-emit with
   a shared "{PAIR:citation_pair_NN}" suffix; atomize() strips this into a
   separate pair_id field per claim instead of leaving it as unlinked text.

4. TYPE-SPECIFIC SUBJECT EXTRACTION FOR CITATION. v1's extraction prompt asked
   for "subject" the same way for every claim type. For CITATION claims this
   caused the LLM to sometimes extract the claim's topic (e.g. "air pollution")
   instead of the citing organization (e.g. "WHO") — a documented failure mode
   downstream in citation_verifier.py's source-name extraction. Fix: the
   extraction prompt now branches — CITATION claims are explicitly told
   "subject" must be the named source, not the topic.

5. SOFT CONTROLLED VOCABULARY FOR "property". v1 let the LLM invent any
   snake_case property string freely, with no connection to what the
   downstream verifiers' PROPERTY_TO_WIKIDATA maps actually cover — the
   confirmed root cause of TEMPORAL/ENTITY_ATTRIBUTE's "unmapped property"
   failures. Fix: the extraction prompt now suggests a list of known-mapped
   property names as a *preference*, not a hard constraint — the LLM can still
   emit a new term if nothing fits, but is nudged toward reusing existing
   mapped vocabulary where the claim genuinely matches one.

New CSV columns versus v1: pair_id, extraction_failed, classification_fallback_used.
"""


import requests
import json
import csv
import re
import time
from datetime import datetime
from sentence_transformers import SentenceTransformer, util

# =============================================================================
# CONFIGURATION
# =============================================================================

OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL_NAME   = "gemma4:31b-cloud"
DRIFT_FLAG_THRESHOLD = 0.50   # below this: flag claim as potentially hallucinated by LLM
DEDUP_THRESHOLD      = 0.90   # above this: treat as duplicate, drop second

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# =============================================================================
# LLM CALL
# =============================================================================

MAX_LLM_RETRIES = 3
BASE_BACKOFF_SECONDS = 2.0

def call_llm(prompt: str, expect_json: bool = False) -> str:
    """
    Single entry point for all LLM calls.
    temperature=0 + seed=42 ensures deterministic, reproducible output.
    expect_json=True activates Ollama grammar-constrained decoding.

    FIX: retries on transient request failures (timeout, connection reset,
    a momentarily-unavailable local Ollama server) with exponential backoff,
    same pattern already used for Wikidata calls in temporal/EA verifiers.
    Without this, a transient failure returns "" — indistinguishable from a
    genuine classification/extraction failure in the new fallback counters,
    which would make a flaky run look like a pipeline quality problem.
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": 8192
        }
    }
    if expect_json:
        payload["format"] = "json"

    for attempt in range(MAX_LLM_RETRIES):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=120)
            response.raise_for_status()
            return response.json()["response"].strip()
        except requests.exceptions.RequestException as e:
            wait = BASE_BACKOFF_SECONDS * (2 ** attempt)
            print(f"  [ERROR] LLM call failed (attempt {attempt+1}/{MAX_LLM_RETRIES}): {e}"
                  + (f" — retrying in {wait:.1f}s" if attempt < MAX_LLM_RETRIES - 1 else " — giving up"))
            if attempt < MAX_LLM_RETRIES - 1:
                time.sleep(wait)

    return ""

# =============================================================================
# STAGE 1 — PARAGRAPH GENERATION
# =============================================================================

def generate_paragraph(question: str) -> str:
    """
    Generate a factually dense paragraph from a question.
    Instructs the LLM to use specific facts, not vague characterizations.
    """
    prompt = (
        f"{question}\n\n"
        "Answer in 5–7 sentences. Use specific, verifiable facts: names, dates, "
        "numbers, causal relationships. Do not use bullet points or headers. "
        "Write in plain declarative sentences only. "
        "Avoid vague phrases like 'revolutionized', 'was influential', or 'is considered'."
    )
    return call_llm(prompt)

# =============================================================================
# STAGE 2 — ATOMIZATION
# =============================================================================

ATOMIZATION_PROMPT_TEMPLATE = """Break the following paragraph into a list of atomic factual claims.

Rules:
- Each claim must express exactly one verifiable fact.
- Each claim must be self-contained: resolve all pronouns (e.g. "He" → full name).
- Only extract objectively verifiable claims: dates, counts, roles, named inventions, causal events, measurements, source attributions.
- Skip subjective characterizations that no source can confirm or deny (e.g. "revolutionized", "was a visionary", "is considered great").
- If a sentence contains a named source AND an embedded fact (e.g. "The WHO states that air pollution causes 7 million deaths"), emit TWO claims:
    (a) "[Source] reported/stated that [specific claim]" — routes to CITATION verifier
    (b) The embedded fact alone — routes to its own type verifier
    Tag BOTH of these two lines with the same pair marker at the end, e.g.:
        "The WHO reported that air pollution causes 7 million deaths annually. {{PAIR:citation_pair_01}}"
        "Air pollution causes approximately 7 million deaths annually. {{PAIR:citation_pair_01}}"
    Use a new pair number (citation_pair_02, _03, ...) for each distinct dual-emit
    in this paragraph. Do NOT add a {{PAIR:...}} tag to claims that aren't part of
    a dual-emit pair.
- Do not merge distinct facts. Do not add facts not present in the paragraph.

Output one claim per line. No numbering, no bullets, no explanation.

Paragraph:
{paragraph}"""

PAIR_TAG_PATTERN = re.compile(r"\s*\{PAIR:([\w-]+)\}\s*$")

def atomize(paragraph: str) -> list[dict]:
    """
    Returns a list of {"text": claim_text, "pair_id": str | None} dicts instead
    of bare strings, so the citation dual-emit pairing (FIX 3) survives into
    downstream processing rather than being silently dropped.
    """
    prompt = ATOMIZATION_PROMPT_TEMPLATE.format(paragraph=paragraph)
    raw = call_llm(prompt)
    lines = [line.strip() for line in raw.split("\n") if line.strip()]

    parsed = []
    for line in lines:
        match = PAIR_TAG_PATTERN.search(line)
        if match:
            pair_id = match.group(1)
            text = PAIR_TAG_PATTERN.sub("", line).strip()
        else:
            pair_id = None
            text = line
        parsed.append({"text": text, "pair_id": pair_id})
    return parsed

# =============================================================================
# STAGE 3 — DEDUPLICATION
# =============================================================================

def deduplicate(claims: list[dict]) -> tuple[list[dict], int]:
    """
    Remove near-duplicate claims using Sentence-BERT cosine similarity.
    Threshold τ = 0.90 per HalDetect essentials spec.
    Returns (deduplicated list, number of claims dropped).
    Both raw and deduplicated counts are retained as metadata.

    Operates on claim dicts ({"text", "pair_id"}) rather than bare strings so
    pair_id survives deduplication — see atomize()'s FIX 3.
    """
    if not claims:
        return [], 0

    texts = [c["text"] for c in claims]
    embeddings = embedder.encode(texts, convert_to_tensor=True)
    kept_indices = [0]

    for i in range(1, len(claims)):
        sims = [util.cos_sim(embeddings[i], embeddings[j]).item() for j in kept_indices]
        if max(sims) < DEDUP_THRESHOLD:
            kept_indices.append(i)

    dropped = len(claims) - len(kept_indices)
    return [claims[i] for i in kept_indices], dropped

# =============================================================================
# STAGE 4 — DRIFT CHECK
# =============================================================================

def compute_drift(paragraph: str, claims: list[dict]) -> list[float]:
    """
    Measures cosine similarity between each claim and the source paragraph.
    Low score = claim may have been paraphrased or hallucinated by LLM during atomization.
    Threshold 0.50: flag for review, do NOT auto-discard.

    Operates on claim dicts ({"text", "pair_id"}) — only the text is embedded.
    """
    para_emb = embedder.encode(paragraph, convert_to_tensor=True)
    scores = []
    for claim in claims:
        claim_emb = embedder.encode(claim["text"], convert_to_tensor=True)
        sim = util.cos_sim(para_emb, claim_emb).item()
        scores.append(round(sim, 3))
    return scores

# =============================================================================
# STAGE 5 — CLASSIFICATION (LLM, FEW-SHOT)
# =============================================================================

# Few-shot examples drawn directly from the 140-claim corpus (ground truth).
# Priority order matches taxonomy rules doc Section 3.1.
CLASSIFICATION_PROMPT_TEMPLATE = """Classify the following claim into exactly one taxonomy type.

PRIORITY ORDER (apply top-down, stop at first match):
1. CITATION     — claim names a specific source (org, report, journal) that made a specific assertion.
                  The source must be named AND the assertion must be attributable to it.
                  "WHO exists" is NOT a CITATION claim. "The WHO reported X" IS.
2. TEMPORAL     — the primary verifiable element is a date, year, or era.
                  A year mentioned only as context (e.g. "during the 1905 crisis") does NOT make it TEMPORAL.
3. QUANTITATIVE — the primary verifiable element is a specific number, percentage, or measurement.
                  Vague quantities ("millions", "many") do NOT qualify.
4. ENTITY_ATTRIBUTE — describes what an entity IS, HAS, or its static role/property.
5. RELATIONAL   — default. Describes what something DOES, CAUSES, HOW it works, or comparative claims.

Examples (deliberately drawn from domains NOT used elsewhere in this pipeline's
test corpus — geography, ecology, astronomy, world history, international
institutions — so the classifier learns the general rule rather than
pattern-matching on Einstein/Tesla/2008-crisis/DNA/Lehman-style phrasing):
- "The Berlin Wall fell on 9 November 1989." → TEMPORAL
- "Mount Everest stands at approximately 8,849 meters above sea level." → QUANTITATIVE
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL
- "UNESCO designated the Great Barrier Reef a World Heritage Site in 1981." → CITATION
- "The blue whale can weigh up to 200 metric tons." → QUANTITATIVE
- "Saturn is a gas giant known for its prominent ring system." → ENTITY_ATTRIBUTE
- "The collapse of the Berlin Wall accelerated German reunification." → RELATIONAL
- "The World Bank's 2023 report states that global extreme poverty stagnated at 8.4%." → CITATION
- "The Ming dynasty's Great Wall sections were built mainly between 1368 and 1644." → TEMPORAL

Note the contrast between two examples already above — same subject area
(the Amazon), different classification, because one is about what something
IS/HAS and the other is about what it DOES/CAUSES:
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE (what it IS)
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL (what it CAUSES)
This is the most common disagreement point — when in doubt, ask "what does X do/cause" (RELATIONAL) vs. "what is X / what does X have" (ENTITY_ATTRIBUTE).

Claim: "{claim}"

Reply with exactly one word — the type label. Nothing else."""

def classify_claim(claim: str) -> tuple[str, bool, str]:
    """
    Returns (claim_type, fallback_used, raw_output).
    FIX 1: on a malformed LLM response, re-prompt once before giving up. If
    still unmatched, tag fallback_used=True and keep the raw output for
    inspection, rather than silently returning RELATIONAL as if it were a
    genuine classification (v1's behavior — indistinguishable from a real
    RELATIONAL result in the output CSV).
    """
    prompt = CLASSIFICATION_PROMPT_TEMPLATE.format(claim=claim)

    for attempt in range(2):  # original attempt + one re-prompt
        raw = call_llm(prompt).upper().strip()
        for cat in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
            if cat in raw:
                return cat, False, raw

    # Both attempts failed to produce a recognizable label — visible now,
    # not silently absorbed into the RELATIONAL count.
    return "RELATIONAL", True, raw

# =============================================================================
# STAGE 6 — STRUCTURED JSON EXTRACTION
# =============================================================================

EXTRACTION_PROMPT_TEMPLATE = """Extract the structured elements from this factual claim. Return valid JSON only.

Claim: "{claim}"
Claim type: "{claim_type}"

Return exactly this structure:
{{
  "claim": "<the original claim verbatim>",
  "subject": "<primary entity the claim is about>",
  "property": "<what is being asserted, e.g. birth_year, nationality, inventor_of, cause_of>",
  "value": "<the asserted value or target>",
  "secondary_context": {{}}
}}

Rules:
- "subject" must be a named entity (person, organization, event, concept).
{subject_rule}
- "property" should be a short snake_case descriptor usable as a Wikidata/DBpedia property label.
  Prefer one of these existing property names if the claim genuinely matches one — this helps
  downstream verification reuse known mappings — but if none of them fit the claim, invent a new
  snake_case term rather than forcing a bad fit:
  {property_vocab}
- "value" should be the specific fact being claimed (a year, a name, a number, a place).
- "secondary_context" holds any additional entities or qualifiers (e.g. location, unit).
- JSON only. No explanation, no preamble."""

CITATION_SUBJECT_RULE = (
    '- Since this claim is type CITATION, "subject" MUST be the named source/organization '
    'itself (e.g. "WHO", "IPCC", "CDC") — NOT the topic of the fact being cited '
    '(e.g. not "air pollution", not "vaccine schedules"). The topic goes in "value" instead.'
)

KNOWN_PROPERTY_VOCAB = [
    "birth_year", "birth_date", "death_year", "death_date", "birth_name", "place_of_birth",
    "employer", "award", "publication_year", "founded_year", "inception_date",
    "dissolution_year", "member_of", "is_specialized_agency_of", "cause_of_death",
    "consists_of", "inventor_of",
]

def extract_structure(claim: str, claim_type: str = "") -> dict:
    """
    FIX 4: branches the "subject" instruction for CITATION claims specifically,
    since the original single instruction caused the LLM to sometimes extract
    the claim's topic instead of the citing organization (a documented failure
    mode in citation_verifier.py's downstream source-name extraction).
    FIX 5: suggests a controlled property vocabulary as a *preference*, not a
    hard constraint — nudges the LLM toward reusing mapped property names
    without forcing bad fits for claims that genuinely need a new term.
    FIX 2: tags extraction_failed=True on a JSON parse failure instead of
    silently returning nulls indistinguishable from a genuinely empty result.
    """
    subject_rule = CITATION_SUBJECT_RULE if claim_type == "CITATION" else ""
    property_vocab = ", ".join(KNOWN_PROPERTY_VOCAB)

    prompt = EXTRACTION_PROMPT_TEMPLATE.format(
        claim=claim, claim_type=claim_type or "UNKNOWN",
        subject_rule=subject_rule, property_vocab=property_vocab
    )
    raw = call_llm(prompt, expect_json=True)

    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        parsed = json.loads(clean)
        parsed["extraction_failed"] = False
        return parsed
    except json.JSONDecodeError:
        return {
            "claim": claim,
            "subject": None,
            "property": None,
            "value": None,
            "secondary_context": {},
            "extraction_failed": True
        }

# =============================================================================
# FULL PIPELINE
# =============================================================================

def process_question(question: str, q_id: str) -> dict:
    print(f"\n[{q_id}] {question}")

    # Stage 1: Generate paragraph
    paragraph = generate_paragraph(question)
    print(f"  Paragraph ({len(paragraph)} chars): {paragraph[:80]}...")

    # Stage 2: Atomize
    raw_claims = atomize(paragraph)
    raw_count = len(raw_claims)
    print(f"  Atomized: {raw_count} claims")

    # Stage 3: Deduplicate
    claims, dropped = deduplicate(raw_claims)
    print(f"  After dedup: {len(claims)} claims ({dropped} dropped)")

    # Stage 4: Drift check (batch)
    drift_scores = compute_drift(paragraph, claims)

    # Stages 5 + 6: Classify and extract per claim
    processed = []
    type_counts = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                   "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}

    for idx, (claim_dict, drift) in enumerate(zip(claims, drift_scores)):
        claim = claim_dict["text"]
        pair_id = claim_dict["pair_id"]
        flagged = drift < DRIFT_FLAG_THRESHOLD

        claim_type, fallback_used, raw_classification = classify_claim(claim)
        structured = extract_structure(claim, claim_type)

        type_counts[claim_type] += 1

        processed.append({
            "q_id":       q_id,
            "claim_id":   f"{q_id}_C{idx+1:02d}",
            "claim":      claim,
            "type":       claim_type,
            "drift":      drift,
            "flagged":    flagged,
            "pair_id":    pair_id or "",
            "subject":    structured.get("subject"),
            "property":   structured.get("property"),
            "value":      structured.get("value"),
            "secondary":  json.dumps(structured.get("secondary_context", {})),
            "extraction_failed":            structured.get("extraction_failed", False),
            "classification_fallback_used": fallback_used,
            # Gold label fields — left blank for manual annotation
            "gold_type":    "",
            "gold_verdict": "",
            "notes":        ""
        })

        status = " ⚠ DRIFT" if flagged else ""
        fallback_note = " ⚠ CLASSIFY-FALLBACK" if fallback_used else ""
        extract_note = " ⚠ EXTRACT-FAILED" if structured.get("extraction_failed") else ""
        print(f"    [{claim_type}] drift={drift}{status}{fallback_note}{extract_note} | {claim[:70]}")

    return {
        "q_id":        q_id,
        "question":    question,
        "paragraph":   paragraph,
        "raw_count":   raw_count,
        "dedup_count": len(claims),
        "dropped":     dropped,
        "type_counts": type_counts,
        "claims":      processed
    }

# =============================================================================
# OUTPUT — CSV FOR MANUAL GOLD LABELING
# =============================================================================

def save_csv(all_results: list[dict], filepath: str):
    """
    Saves all claims to CSV with blank gold_type, gold_verdict, notes columns.
    This is the file you fill in manually for the pilot dataset.
    """
    fieldnames = [
        "q_id", "claim_id", "claim", "type", "drift", "flagged", "pair_id",
        "subject", "property", "value", "secondary",
        "extraction_failed", "classification_fallback_used",
        "gold_type", "gold_verdict", "notes"
    ]
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for result in all_results:
            for claim_row in result["claims"]:
                writer.writerow(claim_row)
    print(f"\nCSV saved to: {filepath}")

def print_summary(all_results: list[dict]):
    print("\n" + "=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)

    total_raw   = sum(r["raw_count"]   for r in all_results)
    total_dedup = sum(r["dedup_count"] for r in all_results)
    total_drop  = sum(r["dropped"]     for r in all_results)
    total_flagged = sum(
        1 for r in all_results for c in r["claims"] if c["flagged"]
    )

    global_types = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                    "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}
    for r in all_results:
        for t, count in r["type_counts"].items():
            global_types[t] += count

    print(f"Questions processed : {len(all_results)}")
    print(f"Raw claims          : {total_raw}")
    print(f"After dedup         : {total_dedup} ({total_drop} dropped)")
    print(f"Drift-flagged       : {total_flagged} (drift < {DRIFT_FLAG_THRESHOLD})")

    print("\n" + "-" * 60)
    print("CUMULATIVE TYPE DISTRIBUTION (all questions)")
    print("-" * 60)
    for t in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
        count = global_types[t]
        pct = (count / total_dedup * 100) if total_dedup else 0
        print(f"  {t:<20} {count:>4}  ({pct:>5.1f}%)")

    print("\n" + "-" * 60)
    print("PER-QUESTION TYPE BREAKDOWN")
    print("-" * 60)
    print(f"{'Q_ID':<6} {'Total':<8} {'CITATION':<10} {'TEMPORAL':<10} {'QUANTITATIVE':<15} {'ENTITY_ATTR':<15} {'RELATIONAL':<12}")
    print("-" * 60)
    for r in all_results:
        q_id = r['q_id']
        total = r['dedup_count']
        citation = r['type_counts'].get('CITATION', 0)
        temporal = r['type_counts'].get('TEMPORAL', 0)
        quantitative = r['type_counts'].get('QUANTITATIVE', 0)
        entity_attr = r['type_counts'].get('ENTITY_ATTRIBUTE', 0)
        relational = r['type_counts'].get('RELATIONAL', 0)
        print(f"{q_id:<6} {total:<8} {citation:<10} {temporal:<10} {quantitative:<15} {entity_attr:<15} {relational:<12}")

    print("\n" + "-" * 60)
    print("SUMMARY STATISTICS")
    print("-" * 60)
    print(f"  Average claims per question : {total_dedup / len(all_results):.1f}")
    print(f"  Deduplication rate          : {(total_drop / total_raw * 100):.1f}% ({total_drop}/{total_raw})")
    print(f"  Drift-flagged rate          : {(total_flagged / total_dedup * 100):.1f}% ({total_flagged}/{total_dedup})")
    print(f"  Most common type            : {max(global_types, key=global_types.get)} ({global_types[max(global_types, key=global_types.get)]} claims)")
    print(f"  Least common type           : {min(global_types, key=global_types.get)} ({global_types[min(global_types, key=global_types.get)]} claims)")

    total_fallback = sum(1 for r in all_results for c in r["claims"] if c["classification_fallback_used"])
    total_extract_failed = sum(1 for r in all_results for c in r["claims"] if c["extraction_failed"])
    print(f"  Classification fallbacks    : {total_fallback}/{total_dedup} "
          f"({(total_fallback / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows' 'type' is NOT a genuine classification, re-check before using")
    print(f"  Extraction failures         : {total_extract_failed}/{total_dedup} "
          f"({(total_extract_failed / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows need re-running, not gold-labeling as UNVERIFIABLE")

# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # Batch 3 ("B3") — new question set, domains deliberately distinct from the
    # existing corpus (no Einstein, Tesla, 2008 crisis, DNA, CRISPR/mRNA, UNSC,
    # Paris Agreement, Cuban Missile Crisis, Marie Curie, internet history) to
    # test generalization rather than reuse familiar phrasing/properties.
    # Not tuned to hit a specific type-distribution percentage — see the note
    # in batch3_questions.py on why that's deliberately not the goal.
    questions = [
        ("B3_01", "What is the history and engineering significance of the Panama Canal?"),
        ("B3_02", "What does the International Energy Agency report about global renewable energy capacity growth?"),
        ("B3_03", "How does coral bleaching occur and what causes it?"),
        ("B3_04", "What are the tallest mountains on each continent and their exact elevations?"),
        ("B3_05", "What does UNESCO's World Heritage Committee consider when designating a new World Heritage Site?"),
        ("B3_06", "What is the Rosetta Stone and why was it significant for understanding ancient Egyptian writing?"),
        ("B3_07", "According to World Athletics, what are the current world records in the men's and women's 100m sprint?"),
        ("B3_08", "What caused the collapse of the Aral Sea and what are its consequences?"),
        ("B3_09", "What does the OECD report about income inequality trends across member countries?"),
        ("B3_10", "What is the structure and function of the human liver?"),
        ("B3_11", "How many satellites does each major space agency currently have in orbit, and which countries lead in launches?"),
        ("B3_12", "What does the FAO report about global fish stock depletion and overfishing?"),
    ]

    all_results = []
    for q_id, question in questions:
        result = process_question(question, q_id)
        all_results.append(result)

    print_summary(all_results)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path  = f"haldetect_claims_{timestamp}.csv"
    save_csv(all_results, csv_path)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7125.52it/s]



[B3_01] What is the history and engineering significance of the Panama Canal?
  Paragraph (559 chars): France began constructing the canal in 1881 under Ferdinand de Lesseps. This att...
  Atomized: 11 claims
  After dedup: 11 claims (0 dropped)
    [TEMPORAL] drift=0.675 | France began constructing the canal in 1881.
    [RELATIONAL] drift=0.597 | Ferdinand de Lesseps led the French construction of the canal.
    [RELATIONAL] drift=0.577 | Yellow fever and malaria caused the French attempt to construct the ca
    [QUANTITATIVE] drift=0.582 | Yellow fever and malaria killed over 20,000 workers during the French 
    [TEMPORAL] drift=0.629 | The United States took over the canal project in 1904.
    [RELATIONAL] drift=0.353 ⚠ DRIFT | The United States supported Panamanian independence from Colombia.
    [RELATIONAL] drift=0.53 | Engineers built the Gatun Dam.
    [ENTITY_ATTRIBUTE] drift=0.562 | The Gatun Dam created the largest man-made lake of its time.
    [QUANTITATIVE] drift=0.441

## LLM generated responses - batch 3

In [2]:
"""
HalDetect Pipeline — Extraction, Deduplication, Classification (v2)
CSE 700 · University of Chittagong · 2026

Pipeline stages implemented here:
    1. Paragraph generation (LLM)

    2. Atomization (LLM) — with subjectivity filter built into prompt
    3. Deduplication (Sentence-BERT, τ = 0.90)
    4. Drift check (Sentence-BERT cosine sim vs source paragraph)
    5. Classification (LLM, few-shot, temperature=0, seed=42)
    6. Structured JSON extraction per claim (LLM)
    7. CSV output for manual gold labeling

Verifiers (TEMPORAL, ENTITY_ATTRIBUTE, CITATION, QUANTITATIVE, RELATIONAL)
are separate modules — not implemented here.

CHANGES FROM v1 (five fixes, all instrumentation/prompt changes — no behavior
change to what a claim ultimately gets classified/extracted as):

1. CLASSIFICATION FAILURE VISIBILITY. v1's classify_claim() silently returned
   "RELATIONAL" whenever the LLM's raw output didn't contain one of the five
   category words verbatim — indistinguishable from a genuine RELATIONAL
   classification. Fix: on failure, re-prompt once; if still unmatched, tag the
   row with classification_fallback_used=True and evidence in
   classification_raw_output, rather than silently defaulting.

2. EXTRACTION FAILURE VISIBILITY. v1's extract_structure() returned
   {subject: None, property: None, value: None} on a JSONDecodeError —
   indistinguishable from a genuinely empty extraction. Fix: tag the row with
   extraction_failed=True instead of silently returning nulls.

3. CITATION DUAL-EMIT PAIRING. v1's atomize() had no mechanism to link a
   source-attribution claim back to its embedded-fact sibling when the
   atomization prompt dual-emits both (per the prompt's own rule). Fix: the
   atomization prompt now asks the LLM to tag both halves of a dual-emit with
   a shared "{PAIR:citation_pair_NN}" suffix; atomize() strips this into a
   separate pair_id field per claim instead of leaving it as unlinked text.

4. TYPE-SPECIFIC SUBJECT EXTRACTION FOR CITATION. v1's extraction prompt asked
   for "subject" the same way for every claim type. For CITATION claims this
   caused the LLM to sometimes extract the claim's topic (e.g. "air pollution")
   instead of the citing organization (e.g. "WHO") — a documented failure mode
   downstream in citation_verifier.py's source-name extraction. Fix: the
   extraction prompt now branches — CITATION claims are explicitly told
   "subject" must be the named source, not the topic.

5. SOFT CONTROLLED VOCABULARY FOR "property". v1 let the LLM invent any
   snake_case property string freely, with no connection to what the
   downstream verifiers' PROPERTY_TO_WIKIDATA maps actually cover — the
   confirmed root cause of TEMPORAL/ENTITY_ATTRIBUTE's "unmapped property"
   failures. Fix: the extraction prompt now suggests a list of known-mapped
   property names as a *preference*, not a hard constraint — the LLM can still
   emit a new term if nothing fits, but is nudged toward reusing existing
   mapped vocabulary where the claim genuinely matches one.

New CSV columns versus v1: pair_id, extraction_failed, classification_fallback_used.
"""


import requests
import json
import csv
import re
import time
from datetime import datetime
from sentence_transformers import SentenceTransformer, util

# =============================================================================
# CONFIGURATION
# =============================================================================

OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL_NAME   = "gemma4:31b-cloud"
DRIFT_FLAG_THRESHOLD = 0.50   # below this: flag claim as potentially hallucinated by LLM
DEDUP_THRESHOLD      = 0.90   # above this: treat as duplicate, drop second

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# =============================================================================
# LLM CALL
# =============================================================================

MAX_LLM_RETRIES = 3
BASE_BACKOFF_SECONDS = 2.0

def call_llm(prompt: str, expect_json: bool = False) -> str:
    """
    Single entry point for all LLM calls.
    temperature=0 + seed=42 ensures deterministic, reproducible output.
    expect_json=True activates Ollama grammar-constrained decoding.

    FIX: retries on transient request failures (timeout, connection reset,
    a momentarily-unavailable local Ollama server) with exponential backoff,
    same pattern already used for Wikidata calls in temporal/EA verifiers.
    Without this, a transient failure returns "" — indistinguishable from a
    genuine classification/extraction failure in the new fallback counters,
    which would make a flaky run look like a pipeline quality problem.
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": 8192
        }
    }
    if expect_json:
        payload["format"] = "json"

    for attempt in range(MAX_LLM_RETRIES):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=120)
            response.raise_for_status()
            return response.json()["response"].strip()
        except requests.exceptions.RequestException as e:
            wait = BASE_BACKOFF_SECONDS * (2 ** attempt)
            print(f"  [ERROR] LLM call failed (attempt {attempt+1}/{MAX_LLM_RETRIES}): {e}"
                  + (f" — retrying in {wait:.1f}s" if attempt < MAX_LLM_RETRIES - 1 else " — giving up"))
            if attempt < MAX_LLM_RETRIES - 1:
                time.sleep(wait)

    return ""

# =============================================================================
# STAGE 1 — PARAGRAPH GENERATION
# =============================================================================

def generate_paragraph(question: str) -> str:
    """
    Generate a factually dense paragraph from a question.
    Instructs the LLM to use specific facts, not vague characterizations.
    """
    prompt = (
        f"{question}\n\n"
        "Answer in 5–7 sentences. Use specific, verifiable facts: names, dates, "
        "numbers, causal relationships. Do not use bullet points or headers. "
        "Write in plain declarative sentences only. "
        "Avoid vague phrases like 'revolutionized', 'was influential', or 'is considered'."
    )
    return call_llm(prompt)

# =============================================================================
# STAGE 2 — ATOMIZATION
# =============================================================================

ATOMIZATION_PROMPT_TEMPLATE = """Break the following paragraph into a list of atomic factual claims.

Rules:
- Each claim must express exactly one verifiable fact.
- Each claim must be self-contained: resolve all pronouns (e.g. "He" → full name).
- Only extract objectively verifiable claims: dates, counts, roles, named inventions, causal events, measurements, source attributions.
- Skip subjective characterizations that no source can confirm or deny (e.g. "revolutionized", "was a visionary", "is considered great").
- If a sentence contains a named source AND an embedded fact (e.g. "The WHO states that air pollution causes 7 million deaths"), emit TWO claims:
    (a) "[Source] reported/stated that [specific claim]" — routes to CITATION verifier
    (b) The embedded fact alone — routes to its own type verifier
    Tag BOTH of these two lines with the same pair marker at the end, e.g.:
        "The WHO reported that air pollution causes 7 million deaths annually. {{PAIR:citation_pair_01}}"
        "Air pollution causes approximately 7 million deaths annually. {{PAIR:citation_pair_01}}"
    Use a new pair number (citation_pair_02, _03, ...) for each distinct dual-emit
    in this paragraph. Do NOT add a {{PAIR:...}} tag to claims that aren't part of
    a dual-emit pair.
- Do not merge distinct facts. Do not add facts not present in the paragraph.

Output one claim per line. No numbering, no bullets, no explanation.

Paragraph:
{paragraph}"""

PAIR_TAG_PATTERN = re.compile(r"\s*\{PAIR:([\w-]+)\}\s*$")

def atomize(paragraph: str) -> list[dict]:
    """
    Returns a list of {"text": claim_text, "pair_id": str | None} dicts instead
    of bare strings, so the citation dual-emit pairing (FIX 3) survives into
    downstream processing rather than being silently dropped.
    """
    prompt = ATOMIZATION_PROMPT_TEMPLATE.format(paragraph=paragraph)
    raw = call_llm(prompt)
    lines = [line.strip() for line in raw.split("\n") if line.strip()]

    parsed = []
    for line in lines:
        match = PAIR_TAG_PATTERN.search(line)
        if match:
            pair_id = match.group(1)
            text = PAIR_TAG_PATTERN.sub("", line).strip()
        else:
            pair_id = None
            text = line
        parsed.append({"text": text, "pair_id": pair_id})
    return parsed

# =============================================================================
# STAGE 3 — DEDUPLICATION
# =============================================================================

def deduplicate(claims: list[dict]) -> tuple[list[dict], int]:
    """
    Remove near-duplicate claims using Sentence-BERT cosine similarity.
    Threshold τ = 0.90 per HalDetect essentials spec.
    Returns (deduplicated list, number of claims dropped).
    Both raw and deduplicated counts are retained as metadata.

    Operates on claim dicts ({"text", "pair_id"}) rather than bare strings so
    pair_id survives deduplication — see atomize()'s FIX 3.
    """
    if not claims:
        return [], 0

    texts = [c["text"] for c in claims]
    embeddings = embedder.encode(texts, convert_to_tensor=True)
    kept_indices = [0]

    for i in range(1, len(claims)):
        sims = [util.cos_sim(embeddings[i], embeddings[j]).item() for j in kept_indices]
        if max(sims) < DEDUP_THRESHOLD:
            kept_indices.append(i)

    dropped = len(claims) - len(kept_indices)
    return [claims[i] for i in kept_indices], dropped

# =============================================================================
# STAGE 4 — DRIFT CHECK
# =============================================================================

def compute_drift(paragraph: str, claims: list[dict]) -> list[float]:
    """
    Measures cosine similarity between each claim and the source paragraph.
    Low score = claim may have been paraphrased or hallucinated by LLM during atomization.
    Threshold 0.50: flag for review, do NOT auto-discard.

    Operates on claim dicts ({"text", "pair_id"}) — only the text is embedded.
    """
    para_emb = embedder.encode(paragraph, convert_to_tensor=True)
    scores = []
    for claim in claims:
        claim_emb = embedder.encode(claim["text"], convert_to_tensor=True)
        sim = util.cos_sim(para_emb, claim_emb).item()
        scores.append(round(sim, 3))
    return scores

# =============================================================================
# STAGE 5 — CLASSIFICATION (LLM, FEW-SHOT)
# =============================================================================

# Few-shot examples drawn directly from the 140-claim corpus (ground truth).
# Priority order matches taxonomy rules doc Section 3.1.
CLASSIFICATION_PROMPT_TEMPLATE = """Classify the following claim into exactly one taxonomy type.

PRIORITY ORDER (apply top-down, stop at first match):
1. CITATION     — claim names a specific source (org, report, journal) that made a specific assertion.
                  The source must be named AND the assertion must be attributable to it.
                  "WHO exists" is NOT a CITATION claim. "The WHO reported X" IS.
2. TEMPORAL     — the primary verifiable element is a date, year, or era.
                  A year mentioned only as context (e.g. "during the 1905 crisis") does NOT make it TEMPORAL.
3. QUANTITATIVE — the primary verifiable element is a specific number, percentage, or measurement.
                  Vague quantities ("millions", "many") do NOT qualify.
4. ENTITY_ATTRIBUTE — describes what an entity IS, HAS, or its static role/property.
5. RELATIONAL   — default. Describes what something DOES, CAUSES, HOW it works, or comparative claims.

Examples (deliberately drawn from domains NOT used elsewhere in this pipeline's
test corpus — geography, ecology, astronomy, world history, international
institutions — so the classifier learns the general rule rather than
pattern-matching on Einstein/Tesla/2008-crisis/DNA/Lehman-style phrasing):
- "The Berlin Wall fell on 9 November 1989." → TEMPORAL
- "Mount Everest stands at approximately 8,849 meters above sea level." → QUANTITATIVE
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL
- "UNESCO designated the Great Barrier Reef a World Heritage Site in 1981." → CITATION
- "The blue whale can weigh up to 200 metric tons." → QUANTITATIVE
- "Saturn is a gas giant known for its prominent ring system." → ENTITY_ATTRIBUTE
- "The collapse of the Berlin Wall accelerated German reunification." → RELATIONAL
- "The World Bank's 2023 report states that global extreme poverty stagnated at 8.4%." → CITATION
- "The Ming dynasty's Great Wall sections were built mainly between 1368 and 1644." → TEMPORAL

Note the contrast between two examples already above — same subject area
(the Amazon), different classification, because one is about what something
IS/HAS and the other is about what it DOES/CAUSES:
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE (what it IS)
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL (what it CAUSES)
This is the most common disagreement point — when in doubt, ask "what does X do/cause" (RELATIONAL) vs. "what is X / what does X have" (ENTITY_ATTRIBUTE).

Claim: "{claim}"

Reply with exactly one word — the type label. Nothing else."""

def classify_claim(claim: str) -> tuple[str, bool, str]:
    """
    Returns (claim_type, fallback_used, raw_output).
    FIX 1: on a malformed LLM response, re-prompt once before giving up. If
    still unmatched, tag fallback_used=True and keep the raw output for
    inspection, rather than silently returning RELATIONAL as if it were a
    genuine classification (v1's behavior — indistinguishable from a real
    RELATIONAL result in the output CSV).
    """
    prompt = CLASSIFICATION_PROMPT_TEMPLATE.format(claim=claim)

    for attempt in range(2):  # original attempt + one re-prompt
        raw = call_llm(prompt).upper().strip()
        for cat in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
            if cat in raw:
                return cat, False, raw

    # Both attempts failed to produce a recognizable label — visible now,
    # not silently absorbed into the RELATIONAL count.
    return "RELATIONAL", True, raw

# =============================================================================
# STAGE 6 — STRUCTURED JSON EXTRACTION
# =============================================================================

EXTRACTION_PROMPT_TEMPLATE = """Extract the structured elements from this factual claim. Return valid JSON only.

Claim: "{claim}"
Claim type: "{claim_type}"

Return exactly this structure:
{{
  "claim": "<the original claim verbatim>",
  "subject": "<primary entity the claim is about>",
  "property": "<what is being asserted, e.g. birth_year, nationality, inventor_of, cause_of>",
  "value": "<the asserted value or target>",
  "secondary_context": {{}}
}}

Rules:
- "subject" must be a named entity (person, organization, event, concept).
{subject_rule}
- "property" should be a short snake_case descriptor usable as a Wikidata/DBpedia property label.
  Prefer one of these existing property names if the claim genuinely matches one — this helps
  downstream verification reuse known mappings — but if none of them fit the claim, invent a new
  snake_case term rather than forcing a bad fit:
  {property_vocab}
- "value" should be the specific fact being claimed (a year, a name, a number, a place).
- "secondary_context" holds any additional entities or qualifiers (e.g. location, unit).
- JSON only. No explanation, no preamble."""

CITATION_SUBJECT_RULE = (
    '- Since this claim is type CITATION, "subject" MUST be the named source/organization '
    'itself (e.g. "WHO", "IPCC", "CDC") — NOT the topic of the fact being cited '
    '(e.g. not "air pollution", not "vaccine schedules"). The topic goes in "value" instead.'
)

KNOWN_PROPERTY_VOCAB = [
    "birth_year", "birth_date", "death_year", "death_date", "birth_name", "place_of_birth",
    "employer", "award", "publication_year", "founded_year", "inception_date",
    "dissolution_year", "member_of", "is_specialized_agency_of", "cause_of_death",
    "consists_of", "inventor_of",
]

def extract_structure(claim: str, claim_type: str = "") -> dict:
    """
    FIX 4: branches the "subject" instruction for CITATION claims specifically,
    since the original single instruction caused the LLM to sometimes extract
    the claim's topic instead of the citing organization (a documented failure
    mode in citation_verifier.py's downstream source-name extraction).
    FIX 5: suggests a controlled property vocabulary as a *preference*, not a
    hard constraint — nudges the LLM toward reusing mapped property names
    without forcing bad fits for claims that genuinely need a new term.
    FIX 2: tags extraction_failed=True on a JSON parse failure instead of
    silently returning nulls indistinguishable from a genuinely empty result.
    """
    subject_rule = CITATION_SUBJECT_RULE if claim_type == "CITATION" else ""
    property_vocab = ", ".join(KNOWN_PROPERTY_VOCAB)

    prompt = EXTRACTION_PROMPT_TEMPLATE.format(
        claim=claim, claim_type=claim_type or "UNKNOWN",
        subject_rule=subject_rule, property_vocab=property_vocab
    )
    raw = call_llm(prompt, expect_json=True)

    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        parsed = json.loads(clean)
        parsed["extraction_failed"] = False
        return parsed
    except json.JSONDecodeError:
        return {
            "claim": claim,
            "subject": None,
            "property": None,
            "value": None,
            "secondary_context": {},
            "extraction_failed": True
        }

# =============================================================================
# FULL PIPELINE
# =============================================================================

def process_question(question: str, q_id: str) -> dict:
    print(f"\n[{q_id}] {question}")

    # Stage 1: Generate paragraph
    paragraph = generate_paragraph(question)
    print(f"  Paragraph ({len(paragraph)} chars): {paragraph[:80]}...")

    # Stage 2: Atomize
    raw_claims = atomize(paragraph)
    raw_count = len(raw_claims)
    print(f"  Atomized: {raw_count} claims")

    # Stage 3: Deduplicate
    claims, dropped = deduplicate(raw_claims)
    print(f"  After dedup: {len(claims)} claims ({dropped} dropped)")

    # Stage 4: Drift check (batch)
    drift_scores = compute_drift(paragraph, claims)

    # Stages 5 + 6: Classify and extract per claim
    processed = []
    type_counts = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                   "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}

    for idx, (claim_dict, drift) in enumerate(zip(claims, drift_scores)):
        claim = claim_dict["text"]
        pair_id = claim_dict["pair_id"]
        flagged = drift < DRIFT_FLAG_THRESHOLD

        claim_type, fallback_used, raw_classification = classify_claim(claim)
        structured = extract_structure(claim, claim_type)

        type_counts[claim_type] += 1

        processed.append({
            "q_id":       q_id,
            "claim_id":   f"{q_id}_C{idx+1:02d}",
            "claim":      claim,
            "type":       claim_type,
            "drift":      drift,
            "flagged":    flagged,
            # FIX: prefix with q_id so pair_id is globally unique across the
            # whole output CSV. atomize()'s numbering ("citation_pair_01", _02,
            # ...) restarts at 1 independently for every paragraph/question —
            # it has no visibility into other questions' atomization calls.
            # Without this prefix, two unrelated dual-emits from different
            # questions that both happened to be "the first pair in their
            # paragraph" would both be tagged bare "citation_pair_01" and
            # collide once concatenated into one file — confirmed happening
            # in the Batch 3 run (B3_02, B3_09, B3_12 all shared this exact
            # string despite being three unrelated facts).
            "pair_id":    f"{q_id}_{pair_id}" if pair_id else "",
            "subject":    structured.get("subject"),
            "property":   structured.get("property"),
            "value":      structured.get("value"),
            "secondary":  json.dumps(structured.get("secondary_context", {})),
            "extraction_failed":            structured.get("extraction_failed", False),
            "classification_fallback_used": fallback_used,
            # Gold label fields — left blank for manual annotation
            "gold_type":    "",
            "gold_verdict": "",
            "notes":        ""
        })

        status = " ⚠ DRIFT" if flagged else ""
        fallback_note = " ⚠ CLASSIFY-FALLBACK" if fallback_used else ""
        extract_note = " ⚠ EXTRACT-FAILED" if structured.get("extraction_failed") else ""
        print(f"    [{claim_type}] drift={drift}{status}{fallback_note}{extract_note} | {claim[:70]}")

    return {
        "q_id":        q_id,
        "question":    question,
        "paragraph":   paragraph,
        "raw_count":   raw_count,
        "dedup_count": len(claims),
        "dropped":     dropped,
        "type_counts": type_counts,
        "claims":      processed
    }

# =============================================================================
# OUTPUT — CSV FOR MANUAL GOLD LABELING
# =============================================================================

def save_csv(all_results: list[dict], filepath: str):
    """
    Saves all claims to CSV with blank gold_type, gold_verdict, notes columns.
    This is the file you fill in manually for the pilot dataset.
    """
    fieldnames = [
        "q_id", "claim_id", "claim", "type", "drift", "flagged", "pair_id",
        "subject", "property", "value", "secondary",
        "extraction_failed", "classification_fallback_used",
        "gold_type", "gold_verdict", "notes"
    ]
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for result in all_results:
            for claim_row in result["claims"]:
                writer.writerow(claim_row)
    print(f"\nCSV saved to: {filepath}")

def print_summary(all_results: list[dict]):
    print("\n" + "=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)

    total_raw   = sum(r["raw_count"]   for r in all_results)
    total_dedup = sum(r["dedup_count"] for r in all_results)
    total_drop  = sum(r["dropped"]     for r in all_results)
    total_flagged = sum(
        1 for r in all_results for c in r["claims"] if c["flagged"]
    )

    global_types = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                    "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}
    for r in all_results:
        for t, count in r["type_counts"].items():
            global_types[t] += count

    print(f"Questions processed : {len(all_results)}")
    print(f"Raw claims          : {total_raw}")
    print(f"After dedup         : {total_dedup} ({total_drop} dropped)")
    print(f"Drift-flagged       : {total_flagged} (drift < {DRIFT_FLAG_THRESHOLD})")

    print("\n" + "-" * 60)
    print("CUMULATIVE TYPE DISTRIBUTION (all questions)")
    print("-" * 60)
    for t in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
        count = global_types[t]
        pct = (count / total_dedup * 100) if total_dedup else 0
        print(f"  {t:<20} {count:>4}  ({pct:>5.1f}%)")

    print("\n" + "-" * 60)
    print("PER-QUESTION TYPE BREAKDOWN")
    print("-" * 60)
    print(f"{'Q_ID':<6} {'Total':<8} {'CITATION':<10} {'TEMPORAL':<10} {'QUANTITATIVE':<15} {'ENTITY_ATTR':<15} {'RELATIONAL':<12}")
    print("-" * 60)
    for r in all_results:
        q_id = r['q_id']
        total = r['dedup_count']
        citation = r['type_counts'].get('CITATION', 0)
        temporal = r['type_counts'].get('TEMPORAL', 0)
        quantitative = r['type_counts'].get('QUANTITATIVE', 0)
        entity_attr = r['type_counts'].get('ENTITY_ATTRIBUTE', 0)
        relational = r['type_counts'].get('RELATIONAL', 0)
        print(f"{q_id:<6} {total:<8} {citation:<10} {temporal:<10} {quantitative:<15} {entity_attr:<15} {relational:<12}")

    print("\n" + "-" * 60)
    print("SUMMARY STATISTICS")
    print("-" * 60)
    print(f"  Average claims per question : {total_dedup / len(all_results):.1f}")
    print(f"  Deduplication rate          : {(total_drop / total_raw * 100):.1f}% ({total_drop}/{total_raw})")
    print(f"  Drift-flagged rate          : {(total_flagged / total_dedup * 100):.1f}% ({total_flagged}/{total_dedup})")
    print(f"  Most common type            : {max(global_types, key=global_types.get)} ({global_types[max(global_types, key=global_types.get)]} claims)")
    print(f"  Least common type           : {min(global_types, key=global_types.get)} ({global_types[min(global_types, key=global_types.get)]} claims)")

    total_fallback = sum(1 for r in all_results for c in r["claims"] if c["classification_fallback_used"])
    total_extract_failed = sum(1 for r in all_results for c in r["claims"] if c["extraction_failed"])
    print(f"  Classification fallbacks    : {total_fallback}/{total_dedup} "
          f"({(total_fallback / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows' 'type' is NOT a genuine classification, re-check before using")
    print(f"  Extraction failures         : {total_extract_failed}/{total_dedup} "
          f"({(total_extract_failed / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows need re-running, not gold-labeling as UNVERIFIABLE")

# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # Batch 3 ("B3") — new question set, domains deliberately distinct from the
    # existing corpus (no Einstein, Tesla, 2008 crisis, DNA, CRISPR/mRNA, UNSC,
    # Paris Agreement, Cuban Missile Crisis, Marie Curie, internet history) to
    # test generalization rather than reuse familiar phrasing/properties.
    # Not tuned to hit a specific type-distribution percentage — see the note
    # in batch3_questions.py on why that's deliberately not the goal.
    questions = [
        ("B3_01", "What is the history and engineering significance of the Panama Canal?"),
        ("B3_02", "What does the International Energy Agency report about global renewable energy capacity growth?"),
        ("B3_03", "How does coral bleaching occur and what causes it?"),
        ("B3_04", "What are the tallest mountains on each continent and their exact elevations?"),
        ("B3_05", "What does UNESCO's World Heritage Committee consider when designating a new World Heritage Site?"),
        ("B3_06", "What is the Rosetta Stone and why was it significant for understanding ancient Egyptian writing?"),
        ("B3_07", "According to World Athletics, what are the current world records in the men's and women's 100m sprint?"),
        ("B3_08", "What caused the collapse of the Aral Sea and what are its consequences?"),
        ("B3_09", "What does the OECD report about income inequality trends across member countries?"),
        ("B3_10", "What is the structure and function of the human liver?"),
        ("B3_11", "How many satellites does each major space agency currently have in orbit, and which countries lead in launches?"),
        ("B3_12", "What does the FAO report about global fish stock depletion and overfishing?"),
    ]

    all_results = []
    for q_id, question in questions:
        result = process_question(question, q_id)
        all_results.append(result)

    print_summary(all_results)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path  = f"haldetect_claims_{timestamp}.csv"
    save_csv(all_results, csv_path)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7116.72it/s]



[B3_01] What is the history and engineering significance of the Panama Canal?
  Paragraph (566 chars): France began construction of the canal in 1881 under Ferdinand de Lesseps. The F...
  Atomized: 11 claims
  After dedup: 11 claims (0 dropped)
    [TEMPORAL] drift=0.703 | France began construction of the canal in 1881.
    [TEMPORAL] drift=0.638 | Ferdinand de Lesseps led the French construction of the canal starting
    [TEMPORAL] drift=0.432 ⚠ DRIFT | The French project failed by 1889.
    [RELATIONAL] drift=0.292 ⚠ DRIFT | Yellow fever caused deaths during the French project.
    [RELATIONAL] drift=0.27 ⚠ DRIFT | Malaria caused deaths during the French project.
    [TEMPORAL] drift=0.684 | The United States took over construction of the canal in 1904.
    [RELATIONAL] drift=0.373 ⚠ DRIFT | The United States supported Panamanian independence from Colombia.
    [QUANTITATIVE] drift=0.458 ⚠ DRIFT | Engineers implemented a lock system to lift ships 85 feet above sea le
    [RELATIONA

### previous code with orphaned-rate

In [4]:
"""
HalDetect Pipeline — Extraction, Deduplication, Classification (v2)
CSE 700 · University of Chittagong · 2026

Pipeline stages implemented here:
    1. Paragraph generation (LLM)
    2. Atomization (LLM) — with subjectivity filter built into prompt
    3. Deduplication (Sentence-BERT, τ = 0.90)
    4. Drift check (Sentence-BERT cosine sim vs source paragraph)
    5. Classification (LLM, few-shot, temperature=0, seed=42)
    6. Structured JSON extraction per claim (LLM)
    7. CSV output for manual gold labeling

Verifiers (TEMPORAL, ENTITY_ATTRIBUTE, CITATION, QUANTITATIVE, RELATIONAL)
are separate modules — not implemented here.

CHANGES FROM v1 (five fixes, all instrumentation/prompt changes — no behavior
change to what a claim ultimately gets classified/extracted as):

1. CLASSIFICATION FAILURE VISIBILITY. v1's classify_claim() silently returned
   "RELATIONAL" whenever the LLM's raw output didn't contain one of the five
   category words verbatim — indistinguishable from a genuine RELATIONAL
   classification. Fix: on failure, re-prompt once; if still unmatched, tag the
   row with classification_fallback_used=True and evidence in
   classification_raw_output, rather than silently defaulting.

2. EXTRACTION FAILURE VISIBILITY. v1's extract_structure() returned
   {subject: None, property: None, value: None} on a JSONDecodeError —
   indistinguishable from a genuinely empty extraction. Fix: tag the row with
   extraction_failed=True instead of silently returning nulls.

3. CITATION DUAL-EMIT PAIRING. v1's atomize() had no mechanism to link a
   source-attribution claim back to its embedded-fact sibling when the
   atomization prompt dual-emits both (per the prompt's own rule). Fix: the
   atomization prompt now asks the LLM to tag both halves of a dual-emit with
   a shared "{PAIR:citation_pair_NN}" suffix; atomize() strips this into a
   separate pair_id field per claim instead of leaving it as unlinked text.

4. TYPE-SPECIFIC SUBJECT EXTRACTION FOR CITATION. v1's extraction prompt asked
   for "subject" the same way for every claim type. For CITATION claims this
   caused the LLM to sometimes extract the claim's topic (e.g. "air pollution")
   instead of the citing organization (e.g. "WHO") — a documented failure mode
   downstream in citation_verifier.py's source-name extraction. Fix: the
   extraction prompt now branches — CITATION claims are explicitly told
   "subject" must be the named source, not the topic.

5. SOFT CONTROLLED VOCABULARY FOR "property". v1 let the LLM invent any
   snake_case property string freely, with no connection to what the
   downstream verifiers' PROPERTY_TO_WIKIDATA maps actually cover — the
   confirmed root cause of TEMPORAL/ENTITY_ATTRIBUTE's "unmapped property"
   failures. Fix: the extraction prompt now suggests a list of known-mapped
   property names as a *preference*, not a hard constraint — the LLM can still
   emit a new term if nothing fits, but is nudged toward reusing existing
   mapped vocabulary where the claim genuinely matches one.

New CSV columns versus v1: pair_id, extraction_failed, classification_fallback_used.
"""


import requests
import json
import csv
import re
import time
from datetime import datetime
from sentence_transformers import SentenceTransformer, util

# =============================================================================
# CONFIGURATION
# =============================================================================

OLLAMA_URL   = "http://localhost:11434/api/generate"
MODEL_NAME   = "gemma4:31b-cloud"
DRIFT_FLAG_THRESHOLD = 0.50   # below this: flag claim as potentially hallucinated by LLM
DEDUP_THRESHOLD      = 0.90   # above this: treat as duplicate, drop second

embedder = SentenceTransformer("all-MiniLM-L6-v2")

# =============================================================================
# LLM CALL
# =============================================================================

MAX_LLM_RETRIES = 3
BASE_BACKOFF_SECONDS = 2.0

def call_llm(prompt: str, expect_json: bool = False) -> str:
    """
    Single entry point for all LLM calls.
    temperature=0 + seed=42 ensures deterministic, reproducible output.
    expect_json=True activates Ollama grammar-constrained decoding.

    FIX: retries on transient request failures (timeout, connection reset,
    a momentarily-unavailable local Ollama server) with exponential backoff,
    same pattern already used for Wikidata calls in temporal/EA verifiers.
    Without this, a transient failure returns "" — indistinguishable from a
    genuine classification/extraction failure in the new fallback counters,
    which would make a flaky run look like a pipeline quality problem.
    """
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0,
            "seed": 42,
            "num_ctx": 8192
        }
    }
    if expect_json:
        payload["format"] = "json"

    for attempt in range(MAX_LLM_RETRIES):
        try:
            response = requests.post(OLLAMA_URL, json=payload, timeout=120)
            response.raise_for_status()
            return response.json()["response"].strip()
        except requests.exceptions.RequestException as e:
            wait = BASE_BACKOFF_SECONDS * (2 ** attempt)
            print(f"  [ERROR] LLM call failed (attempt {attempt+1}/{MAX_LLM_RETRIES}): {e}"
                  + (f" — retrying in {wait:.1f}s" if attempt < MAX_LLM_RETRIES - 1 else " — giving up"))
            if attempt < MAX_LLM_RETRIES - 1:
                time.sleep(wait)

    return ""

# =============================================================================
# STAGE 1 — PARAGRAPH GENERATION
# =============================================================================

def generate_paragraph(question: str) -> str:
    """
    Generate a factually dense paragraph from a question.
    Instructs the LLM to use specific facts, not vague characterizations.
    """
    prompt = (
        f"{question}\n\n"
        "Answer in 5–7 sentences. Use specific, verifiable facts: names, dates, "
        "numbers, causal relationships. Do not use bullet points or headers. "
        "Write in plain declarative sentences only. "
        "Avoid vague phrases like 'revolutionized', 'was influential', or 'is considered'."
    )
    return call_llm(prompt)

# =============================================================================
# STAGE 2 — ATOMIZATION
# =============================================================================

ATOMIZATION_PROMPT_TEMPLATE = """Break the following paragraph into a list of atomic factual claims.

Rules:
- Each claim must express exactly one verifiable fact.
- Each claim must be self-contained: resolve all pronouns (e.g. "He" → full name).
- Only extract objectively verifiable claims: dates, counts, roles, named inventions, causal events, measurements, source attributions.
- Skip subjective characterizations that no source can confirm or deny (e.g. "revolutionized", "was a visionary", "is considered great").
- If a sentence contains a named source AND an embedded fact (e.g. "The WHO states that air pollution causes 7 million deaths"), emit TWO claims:
    (a) "[Source] reported/stated that [specific claim]" — routes to CITATION verifier
    (b) The embedded fact alone — routes to its own type verifier
    Tag BOTH of these two lines with the same pair marker at the end, e.g.:
        "The WHO reported that air pollution causes 7 million deaths annually. {{PAIR:citation_pair_01}}"
        "Air pollution causes approximately 7 million deaths annually. {{PAIR:citation_pair_01}}"
    Use a new pair number (citation_pair_02, _03, ...) for each distinct dual-emit
    in this paragraph. Do NOT add a {{PAIR:...}} tag to claims that aren't part of
    a dual-emit pair.
- Do not merge distinct facts. Do not add facts not present in the paragraph.

Output one claim per line. No numbering, no bullets, no explanation.

Paragraph:
{paragraph}"""

PAIR_TAG_PATTERN = re.compile(r"\s*\{PAIR:([\w-]+)\}\s*$")

def atomize(paragraph: str) -> list[dict]:
    """
    Returns a list of {"text": claim_text, "pair_id": str | None} dicts instead
    of bare strings, so the citation dual-emit pairing (FIX 3) survives into
    downstream processing rather than being silently dropped.
    """
    prompt = ATOMIZATION_PROMPT_TEMPLATE.format(paragraph=paragraph)
    raw = call_llm(prompt)
    lines = [line.strip() for line in raw.split("\n") if line.strip()]

    parsed = []
    for line in lines:
        match = PAIR_TAG_PATTERN.search(line)
        if match:
            pair_id = match.group(1)
            text = PAIR_TAG_PATTERN.sub("", line).strip()
        else:
            pair_id = None
            text = line
        parsed.append({"text": text, "pair_id": pair_id})
    return parsed

# =============================================================================
# STAGE 3 — DEDUPLICATION
# =============================================================================

def deduplicate(claims: list[dict]) -> tuple[list[dict], int]:
    """
    Remove near-duplicate claims using Sentence-BERT cosine similarity.
    Threshold τ = 0.90 per HalDetect essentials spec.
    Returns (deduplicated list, number of claims dropped).
    Both raw and deduplicated counts are retained as metadata.

    Operates on claim dicts ({"text", "pair_id"}) rather than bare strings so
    pair_id survives deduplication — see atomize()'s FIX 3.
    """
    if not claims:
        return [], 0

    texts = [c["text"] for c in claims]
    embeddings = embedder.encode(texts, convert_to_tensor=True)
    kept_indices = [0]

    for i in range(1, len(claims)):
        sims = [util.cos_sim(embeddings[i], embeddings[j]).item() for j in kept_indices]
        if max(sims) < DEDUP_THRESHOLD:
            kept_indices.append(i)

    dropped = len(claims) - len(kept_indices)
    return [claims[i] for i in kept_indices], dropped

# =============================================================================
# STAGE 4 — DRIFT CHECK
# =============================================================================

def compute_drift(paragraph: str, claims: list[dict]) -> list[float]:
    """
    Measures cosine similarity between each claim and the source paragraph.
    Low score = claim may have been paraphrased or hallucinated by LLM during atomization.
    Threshold 0.50: flag for review, do NOT auto-discard.

    Operates on claim dicts ({"text", "pair_id"}) — only the text is embedded.
    """
    para_emb = embedder.encode(paragraph, convert_to_tensor=True)
    scores = []
    for claim in claims:
        claim_emb = embedder.encode(claim["text"], convert_to_tensor=True)
        sim = util.cos_sim(para_emb, claim_emb).item()
        scores.append(round(sim, 3))
    return scores

# =============================================================================
# STAGE 5 — CLASSIFICATION (LLM, FEW-SHOT)
# =============================================================================

# Few-shot examples drawn directly from the 140-claim corpus (ground truth).
# Priority order matches taxonomy rules doc Section 3.1.
CLASSIFICATION_PROMPT_TEMPLATE = """Classify the following claim into exactly one taxonomy type.

PRIORITY ORDER (apply top-down, stop at first match):
1. CITATION     — claim names a specific source (org, report, journal) that made a specific assertion.
                  The source must be named AND the assertion must be attributable to it.
                  "WHO exists" is NOT a CITATION claim. "The WHO reported X" IS.
2. TEMPORAL     — the primary verifiable element is a date, year, or era.
                  A year mentioned only as context (e.g. "during the 1905 crisis") does NOT make it TEMPORAL.
3. QUANTITATIVE — the primary verifiable element is a specific number, percentage, or measurement.
                  Vague quantities ("millions", "many") do NOT qualify.
4. ENTITY_ATTRIBUTE — describes what an entity IS, HAS, or its static role/property.
5. RELATIONAL   — default. Describes what something DOES, CAUSES, HOW it works, or comparative claims.

Examples (deliberately drawn from domains NOT used elsewhere in this pipeline's
test corpus — geography, ecology, astronomy, world history, international
institutions — so the classifier learns the general rule rather than
pattern-matching on Einstein/Tesla/2008-crisis/DNA/Lehman-style phrasing):
- "The Berlin Wall fell on 9 November 1989." → TEMPORAL
- "Mount Everest stands at approximately 8,849 meters above sea level." → QUANTITATIVE
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL
- "UNESCO designated the Great Barrier Reef a World Heritage Site in 1981." → CITATION
- "The blue whale can weigh up to 200 metric tons." → QUANTITATIVE
- "Saturn is a gas giant known for its prominent ring system." → ENTITY_ATTRIBUTE
- "The collapse of the Berlin Wall accelerated German reunification." → RELATIONAL
- "The World Bank's 2023 report states that global extreme poverty stagnated at 8.4%." → CITATION
- "The Ming dynasty's Great Wall sections were built mainly between 1368 and 1644." → TEMPORAL

Note the contrast between two examples already above — same subject area
(the Amazon), different classification, because one is about what something
IS/HAS and the other is about what it DOES/CAUSES:
- "The Amazon River is the largest river by discharge volume in the world." → ENTITY_ATTRIBUTE (what it IS)
- "Deforestation in the Amazon basin reduces regional rainfall by disrupting the water cycle." → RELATIONAL (what it CAUSES)
This is the most common disagreement point — when in doubt, ask "what does X do/cause" (RELATIONAL) vs. "what is X / what does X have" (ENTITY_ATTRIBUTE).

Claim: "{claim}"

Reply with exactly one word — the type label. Nothing else."""

def classify_claim(claim: str) -> tuple[str, bool, str]:
    """
    Returns (claim_type, fallback_used, raw_output).
    FIX 1: on a malformed LLM response, re-prompt once before giving up. If
    still unmatched, tag fallback_used=True and keep the raw output for
    inspection, rather than silently returning RELATIONAL as if it were a
    genuine classification (v1's behavior — indistinguishable from a real
    RELATIONAL result in the output CSV).
    """
    prompt = CLASSIFICATION_PROMPT_TEMPLATE.format(claim=claim)

    for attempt in range(2):  # original attempt + one re-prompt
        raw = call_llm(prompt).upper().strip()
        for cat in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
            if cat in raw:
                return cat, False, raw

    # Both attempts failed to produce a recognizable label — visible now,
    # not silently absorbed into the RELATIONAL count.
    return "RELATIONAL", True, raw

# =============================================================================
# STAGE 6 — STRUCTURED JSON EXTRACTION
# =============================================================================

EXTRACTION_PROMPT_TEMPLATE = """Extract the structured elements from this factual claim. Return valid JSON only.

Claim: "{claim}"
Claim type: "{claim_type}"

Return exactly this structure:
{{
  "claim": "<the original claim verbatim>",
  "subject": "<primary entity the claim is about>",
  "property": "<what is being asserted, e.g. birth_year, nationality, inventor_of, cause_of>",
  "value": "<the asserted value or target>",
  "secondary_context": {{}}
}}

Rules:
- "subject" must be a named entity (person, organization, event, concept).
{subject_rule}
- "property" should be a short snake_case descriptor usable as a Wikidata/DBpedia property label.
  Prefer one of these existing property names if the claim genuinely matches one — this helps
  downstream verification reuse known mappings — but if none of them fit the claim, invent a new
  snake_case term rather than forcing a bad fit:
  {property_vocab}
- "value" should be the specific fact being claimed (a year, a name, a number, a place).
- "secondary_context" holds any additional entities or qualifiers (e.g. location, unit).
- JSON only. No explanation, no preamble."""

CITATION_SUBJECT_RULE = (
    '- Since this claim is type CITATION, "subject" MUST be the named source/organization '
    'itself (e.g. "WHO", "IPCC", "CDC") — NOT the topic of the fact being cited '
    '(e.g. not "air pollution", not "vaccine schedules"). The topic goes in "value" instead.'
)

KNOWN_PROPERTY_VOCAB = [
    "birth_year", "birth_date", "death_year", "death_date", "birth_name", "place_of_birth",
    "employer", "award", "publication_year", "founded_year", "inception_date",
    "dissolution_year", "member_of", "is_specialized_agency_of", "cause_of_death",
    "consists_of", "inventor_of",
]

def extract_structure(claim: str, claim_type: str = "") -> dict:
    """
    FIX 4: branches the "subject" instruction for CITATION claims specifically,
    since the original single instruction caused the LLM to sometimes extract
    the claim's topic instead of the citing organization (a documented failure
    mode in citation_verifier.py's downstream source-name extraction).
    FIX 5: suggests a controlled property vocabulary as a *preference*, not a
    hard constraint — nudges the LLM toward reusing mapped property names
    without forcing bad fits for claims that genuinely need a new term.
    FIX 2: tags extraction_failed=True on a JSON parse failure instead of
    silently returning nulls indistinguishable from a genuinely empty result.
    """
    subject_rule = CITATION_SUBJECT_RULE if claim_type == "CITATION" else ""
    property_vocab = ", ".join(KNOWN_PROPERTY_VOCAB)

    prompt = EXTRACTION_PROMPT_TEMPLATE.format(
        claim=claim, claim_type=claim_type or "UNKNOWN",
        subject_rule=subject_rule, property_vocab=property_vocab
    )
    raw = call_llm(prompt, expect_json=True)

    try:
        clean = re.sub(r"```json|```", "", raw).strip()
        parsed = json.loads(clean)
        parsed["extraction_failed"] = False
        return parsed
    except json.JSONDecodeError:
        return {
            "claim": claim,
            "subject": None,
            "property": None,
            "value": None,
            "secondary_context": {},
            "extraction_failed": True
        }

# =============================================================================
# FULL PIPELINE
# =============================================================================

def process_question(question: str, q_id: str) -> dict:
    print(f"\n[{q_id}] {question}")

    # Stage 1: Generate paragraph
    paragraph = generate_paragraph(question)
    print(f"  Paragraph ({len(paragraph)} chars): {paragraph[:80]}...")

    # Stage 2: Atomize
    raw_claims = atomize(paragraph)
    raw_count = len(raw_claims)
    print(f"  Atomized: {raw_count} claims")

    # Stage 3: Deduplicate
    claims, dropped = deduplicate(raw_claims)
    print(f"  After dedup: {len(claims)} claims ({dropped} dropped)")

    # Stage 4: Drift check (batch)
    drift_scores = compute_drift(paragraph, claims)

    # Count how many times each pair_id appears in this question's surviving
    # claims (post-dedup). A genuine dual-emit should produce >=2 members —
    # exactly 1 means either the LLM never emitted/tagged the sibling, or
    # deduplicate() removed it as a near-duplicate (the two halves of a
    # dual-emit, e.g. "WHO reported that X" vs "X", can be similar enough
    # in wording to cross DEDUP_THRESHOLD). Either way, this pair can't be
    # used for source-vs-fact agreement analysis, so it needs to be visible
    # rather than silently look like a valid pair.
    pair_id_counts = {}
    for c in claims:
        if c["pair_id"]:
            pair_id_counts[c["pair_id"]] = pair_id_counts.get(c["pair_id"], 0) + 1

    # Stages 5 + 6: Classify and extract per claim
    processed = []
    type_counts = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                   "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}

    for idx, (claim_dict, drift) in enumerate(zip(claims, drift_scores)):
        claim = claim_dict["text"]
        pair_id = claim_dict["pair_id"]
        pair_incomplete = bool(pair_id) and pair_id_counts.get(pair_id, 0) < 2
        flagged = drift < DRIFT_FLAG_THRESHOLD

        claim_type, fallback_used, raw_classification = classify_claim(claim)
        structured = extract_structure(claim, claim_type)

        type_counts[claim_type] += 1

        processed.append({
            "q_id":       q_id,
            "claim_id":   f"{q_id}_C{idx+1:02d}",
            "claim":      claim,
            "type":       claim_type,
            "drift":      drift,
            "flagged":    flagged,
            # FIX: prefix with q_id so pair_id is globally unique across the
            # whole output CSV. atomize()'s numbering ("citation_pair_01", _02,
            # ...) restarts at 1 independently for every paragraph/question —
            # it has no visibility into other questions' atomization calls.
            # Without this prefix, two unrelated dual-emits from different
            # questions that both happened to be "the first pair in their
            # paragraph" would both be tagged bare "citation_pair_01" and
            # collide once concatenated into one file — confirmed happening
            # in the Batch 3 run (B3_02, B3_09, B3_12 all shared this exact
            # string despite being three unrelated facts).
            "pair_id":    f"{q_id}_{pair_id}" if pair_id else "",
            "pair_incomplete": pair_incomplete,
            "subject":    structured.get("subject"),
            "property":   structured.get("property"),
            "value":      structured.get("value"),
            "secondary":  json.dumps(structured.get("secondary_context", {})),
            "extraction_failed":            structured.get("extraction_failed", False),
            "classification_fallback_used": fallback_used,
            # Gold label fields — left blank for manual annotation
            "gold_type":    "",
            "gold_verdict": "",
            "notes":        ""
        })

        status = " ⚠ DRIFT" if flagged else ""
        fallback_note = " ⚠ CLASSIFY-FALLBACK" if fallback_used else ""
        extract_note = " ⚠ EXTRACT-FAILED" if structured.get("extraction_failed") else ""
        pair_note = " ⚠ ORPHANED-PAIR" if pair_incomplete else ""
        print(f"    [{claim_type}] drift={drift}{status}{fallback_note}{extract_note}{pair_note} | {claim[:70]}")

    return {
        "q_id":        q_id,
        "question":    question,
        "paragraph":   paragraph,
        "raw_count":   raw_count,
        "dedup_count": len(claims),
        "dropped":     dropped,
        "type_counts": type_counts,
        "claims":      processed
    }

# =============================================================================
# OUTPUT — CSV FOR MANUAL GOLD LABELING
# =============================================================================

def save_csv(all_results: list[dict], filepath: str):
    """
    Saves all claims to CSV with blank gold_type, gold_verdict, notes columns.
    This is the file you fill in manually for the pilot dataset.
    """
    fieldnames = [
        "q_id", "claim_id", "claim", "type", "drift", "flagged", "pair_id", "pair_incomplete",
        "subject", "property", "value", "secondary",
        "extraction_failed", "classification_fallback_used",
        "gold_type", "gold_verdict", "notes"
    ]
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for result in all_results:
            for claim_row in result["claims"]:
                writer.writerow(claim_row)
    print(f"\nCSV saved to: {filepath}")

def print_summary(all_results: list[dict]):
    print("\n" + "=" * 60)
    print("PIPELINE SUMMARY")
    print("=" * 60)

    total_raw   = sum(r["raw_count"]   for r in all_results)
    total_dedup = sum(r["dedup_count"] for r in all_results)
    total_drop  = sum(r["dropped"]     for r in all_results)
    total_flagged = sum(
        1 for r in all_results for c in r["claims"] if c["flagged"]
    )

    global_types = {"CITATION": 0, "TEMPORAL": 0, "QUANTITATIVE": 0,
                    "ENTITY_ATTRIBUTE": 0, "RELATIONAL": 0}
    for r in all_results:
        for t, count in r["type_counts"].items():
            global_types[t] += count

    print(f"Questions processed : {len(all_results)}")
    print(f"Raw claims          : {total_raw}")
    print(f"After dedup         : {total_dedup} ({total_drop} dropped)")
    print(f"Drift-flagged       : {total_flagged} (drift < {DRIFT_FLAG_THRESHOLD})")

    print("\n" + "-" * 60)
    print("CUMULATIVE TYPE DISTRIBUTION (all questions)")
    print("-" * 60)
    for t in ["CITATION", "TEMPORAL", "QUANTITATIVE", "ENTITY_ATTRIBUTE", "RELATIONAL"]:
        count = global_types[t]
        pct = (count / total_dedup * 100) if total_dedup else 0
        print(f"  {t:<20} {count:>4}  ({pct:>5.1f}%)")

    print("\n" + "-" * 60)
    print("PER-QUESTION TYPE BREAKDOWN")
    print("-" * 60)
    print(f"{'Q_ID':<6} {'Total':<8} {'CITATION':<10} {'TEMPORAL':<10} {'QUANTITATIVE':<15} {'ENTITY_ATTR':<15} {'RELATIONAL':<12}")
    print("-" * 60)
    for r in all_results:
        q_id = r['q_id']
        total = r['dedup_count']
        citation = r['type_counts'].get('CITATION', 0)
        temporal = r['type_counts'].get('TEMPORAL', 0)
        quantitative = r['type_counts'].get('QUANTITATIVE', 0)
        entity_attr = r['type_counts'].get('ENTITY_ATTRIBUTE', 0)
        relational = r['type_counts'].get('RELATIONAL', 0)
        print(f"{q_id:<6} {total:<8} {citation:<10} {temporal:<10} {quantitative:<15} {entity_attr:<15} {relational:<12}")

    print("\n" + "-" * 60)
    print("SUMMARY STATISTICS")
    print("-" * 60)
    print(f"  Average claims per question : {total_dedup / len(all_results):.1f}")
    print(f"  Deduplication rate          : {(total_drop / total_raw * 100):.1f}% ({total_drop}/{total_raw})")
    print(f"  Drift-flagged rate          : {(total_flagged / total_dedup * 100):.1f}% ({total_flagged}/{total_dedup})")
    print(f"  Most common type            : {max(global_types, key=global_types.get)} ({global_types[max(global_types, key=global_types.get)]} claims)")
    print(f"  Least common type           : {min(global_types, key=global_types.get)} ({global_types[min(global_types, key=global_types.get)]} claims)")

    total_fallback = sum(1 for r in all_results for c in r["claims"] if c["classification_fallback_used"])
    total_extract_failed = sum(1 for r in all_results for c in r["claims"] if c["extraction_failed"])
    print(f"  Classification fallbacks    : {total_fallback}/{total_dedup} "
          f"({(total_fallback / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows' 'type' is NOT a genuine classification, re-check before using")
    print(f"  Extraction failures         : {total_extract_failed}/{total_dedup} "
          f"({(total_extract_failed / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these rows need re-running, not gold-labeling as UNVERIFIABLE")

    total_orphaned = sum(1 for r in all_results for c in r["claims"] if c["pair_incomplete"])
    print(f"  Orphaned citation pairs     : {total_orphaned}/{total_dedup} "
          f"({(total_orphaned / total_dedup * 100) if total_dedup else 0:.1f}%) — "
          f"these have a pair_id but no matching sibling; can't be used for "
          f"source-vs-fact agreement analysis")

# =============================================================================
# ENTRY POINT
# =============================================================================

if __name__ == "__main__":

    # Batch 3 ("B3") — new question set, domains deliberately distinct from the
    # existing corpus (no Einstein, Tesla, 2008 crisis, DNA, CRISPR/mRNA, UNSC,
    # Paris Agreement, Cuban Missile Crisis, Marie Curie, internet history) to
    # test generalization rather than reuse familiar phrasing/properties.
    # Not tuned to hit a specific type-distribution percentage — see the note
    # in batch3_questions.py on why that's deliberately not the goal.
    questions = [
        ("B3_01", "What is the history and engineering significance of the Panama Canal?"),
        ("B3_02", "What does the International Energy Agency report about global renewable energy capacity growth?"),
        ("B3_03", "How does coral bleaching occur and what causes it?"),
        ("B3_04", "What are the tallest mountains on each continent and their exact elevations?"),
        ("B3_05", "What does UNESCO's World Heritage Committee consider when designating a new World Heritage Site?"),
        ("B3_06", "What is the Rosetta Stone and why was it significant for understanding ancient Egyptian writing?"),
        ("B3_07", "According to World Athletics, what are the current world records in the men's and women's 100m sprint?"),
        ("B3_08", "What caused the collapse of the Aral Sea and what are its consequences?"),
        ("B3_09", "What does the OECD report about income inequality trends across member countries?"),
        ("B3_10", "What is the structure and function of the human liver?"),
        ("B3_11", "How many satellites does each major space agency currently have in orbit, and which countries lead in launches?"),
        ("B3_12", "What does the FAO report about global fish stock depletion and overfishing?"),
    ]

    all_results = []
    for q_id, question in questions:
        result = process_question(question, q_id)
        all_results.append(result)

    print_summary(all_results)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    csv_path  = f"haldetect_claims_{timestamp}.csv"
    save_csv(all_results, csv_path)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6424.66it/s]



[B3_01] What is the history and engineering significance of the Panama Canal?
  Paragraph (686 chars): France began construction of the Panama Canal in 1881 under Ferdinand de Lesseps...
  Atomized: 12 claims
  After dedup: 12 claims (0 dropped)
    [TEMPORAL] drift=0.759 | France began construction of the Panama Canal in 1881.
    [RELATIONAL] drift=0.692 | Ferdinand de Lesseps led the French construction of the Panama Canal.
    [TEMPORAL] drift=0.746 | The French effort to build the Panama Canal failed by 1889.
    [QUANTITATIVE] drift=0.692 | Yellow fever and malaria caused over 20,000 deaths during the French e
    [TEMPORAL] drift=0.711 | The United States took over the Panama Canal project in 1904.
    [RELATIONAL] drift=0.44 ⚠ DRIFT | The United States supported Panamanian independence from Colombia.
    [RELATIONAL] drift=0.673 | Engineers replaced the sea-level plan for the Panama Canal with a lock
    [RELATIONAL] drift=0.636 | Engineers created Gatun Lake to manage the ter